# pandas 03. 集約・重複・並べ替え

`groupby` の引数、`first`/`last` の落とし穴、「キーごとに1行選ぶ」の書き方3種。

**quest-01 でハマるのはここ。** 特に 3-2 と 4-1。

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

NULLISH = {"NULL", "N/A", "-", ""}

def load():
    df = pd.read_csv("/data/orders.csv", dtype=str, keep_default_na=False)
    df = df.map(lambda s: None if str(s).strip() in NULLISH else s)
    df["qty"] = pd.to_numeric(df["qty"]).astype("Int64")
    df["amount"] = pd.to_numeric(df["amount"].str.replace(",", "", regex=False)).astype("Int64")
    df["region"] = df["region"].str.lower()
    return df

df = load()
df

---
## 1. groupby の基本

### 1-1. as_index

グループキーを index にするか、列として残すか。

**実行する前に、違いを予想する。**

In [ ]:
# A: as_index=True (既定)
a = df.groupby("region")["amount"].sum()
print(type(a).__name__)
print(a.index.name)
a

In [ ]:
# B: as_index=False
b = df.groupby("region", as_index=False)["amount"].sum()
print(type(b).__name__)
b

<details>
<summary>何が起きたか</summary>

- `A` は **Series**。`region` が index になる
- `B` は **DataFrame**。`region` が普通の列として残る

index に入ると `df["region"]` で触れなくなり、後続の `merge` や
`to_parquet` で扱いづらい。**下流に渡すなら `as_index=False`**。

`A` から `B` にするには `.reset_index()`。結果は同じ。

</details>

### 1-2. dropna

グループキーが欠損している行はどうなるか。

**実行する前に、違いを予想する。**

In [ ]:
# A: dropna=True (既定)
a = df.groupby("customer_id", as_index=False)["amount"].sum()
print("行数:", len(a), " 合計:", a["amount"].sum())
a

In [ ]:
# B: dropna=False
b = df.groupby("customer_id", as_index=False, dropna=False)["amount"].sum()
print("行数:", len(b), " 合計:", b["amount"].sum())
b

<details>
<summary>何が起きたか</summary>

**既定ではキーが欠損している行は黙って消える。** `O-010` の
`customer_id` が欠損しているので、`A` にはその1960円が入っていない。

集約の前後で合計が変わっていないかを確かめる癖をつける。

```python
print(df["amount"].sum(), a["amount"].sum())
```

「合計が合わない」の原因の上位がこれ。エラーは出ないので、
**数えないと気づけない**。SQL の `GROUP BY` は NULL も1グループとして扱うので、
移植するときも差が出る。

</details>

### 1-3. size と count

行数の数え方。欠損の扱いが違う。

**実行する前に、違いを予想する。**

In [ ]:
# A: size
a = df.groupby("region").size()
print(a)
print("合計:", a.sum())

In [ ]:
# B: count
b = df.groupby("region")["amount"].count()
print(b)
print("合計:", b.sum())

<details>
<summary>何が起きたか</summary>

- `size` は**行数**。欠損も数える
- `count` は**非欠損の数**。`O-006` の `amount` が欠損なので north が1少ない

SQL の `COUNT(*)` が `size`、`COUNT(col)` が `count` に対応する。

「件数」と言われたらどちらか必ず確認する。
**注文件数なら `size`、金額が入っている注文の件数なら `count`。**

</details>

### 1-4. agg の書き方

複数の集約をまとめる。書き方が3通りある。

**実行する前に、違いを予想する。**

In [ ]:
# A: 辞書で指定
a = df.groupby("region", as_index=False).agg({"amount": "sum", "qty": "max"})
a

In [ ]:
# B: 名前付きで指定
b = df.groupby("region", as_index=False).agg(
    total_amount=("amount", "sum"),
    max_qty=("qty", "max"),
    n_orders=("order_id", "size"),
)
b

<details>
<summary>何が起きたか</summary>

`B`(named aggregation)のほうが良い。理由は2つ。

- **出力の列名を自分で決められる。** `A` は元の列名のままなので、
  同じ列に2つの集約をかけると列名が衝突する
- 同じ列に複数の集約をかけられる

```python
df.groupby("region").agg(
    total=("amount", "sum"),
    avg=("amount", "mean"),      # 同じ列に2つ
)
```

集約関数は文字列(`"sum"`)でも関数(`np.sum`、自作の lambda)でも渡せる。
**文字列で足りるなら文字列**のほうが速い(内部の高速な実装が使われる)。

</details>

---
## 2. transform — 集約値を元の行に配る

### 2-1. agg と transform

「地域ごとの合計」を、集計表として欲しいのか、元の行に付けたいのか。

**実行する前に、違いを予想する。**

In [ ]:
# A: agg (行が減る)
a = df.groupby("region")["amount"].sum()
print("行数:", len(a))
a

In [ ]:
# B: transform (行が減らない)
b = df.assign(region_total=df.groupby("region")["amount"].transform("sum"))
print("行数:", len(b))
b[["order_id", "region", "amount", "region_total"]]

<details>
<summary>何が起きたか</summary>

`transform` は**元と同じ長さ**の結果を返し、各行にそのグループの値を配る。

比率や偏差を出すときに使う。

```python
df["share"] = df["amount"] / df.groupby("region")["amount"].transform("sum")
```

`agg` してから `merge` で戻す、と同じことを1行でやっている。
**`merge` を書きたくなったら、まず `transform` で済まないか考える。**

</details>

---
## 3. キーごとに1行選ぶ

**ここが本題。** 「order_id ごとに最新の1行」のような処理は、
書き方によって**結果が変わる**。

In [ ]:
# 「同じ order_id が複数ある」状況を作る。O-001 の訂正が後から届いた想定。
# 訂正のほうが新しい (ingested_at が後) が、amount が欠損している。
dup = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 2400, "status": "completed"},
    {"order_id": "O-001", "ingested_at": "2024-02-02 10:00", "amount": None,  "status": "cancelled"},
    {"order_id": "O-002", "ingested_at": "2024-02-01 10:00", "amount": 980,   "status": "completed"},
])
dup["amount"] = dup["amount"].astype("Int64")
dup

### 3-1. groupby().last() は「最後の行」ではない

「order_id ごとに ingested_at が最新の行」を取りたい。2つの書き方を比べる。

**実行する前に、違いを予想する。**

In [ ]:
# A: groupby().last()
a = dup.sort_values("ingested_at").groupby("order_id", as_index=False).last()
a

In [ ]:
# B: drop_duplicates(keep='last')
b = dup.sort_values("ingested_at").drop_duplicates(subset="order_id", keep="last")
b

<details>
<summary>何が起きたか</summary>

**O-001 の行を見比べる。**

`A` の `amount` は **2400**。最新行の `amount` は欠損なのに、値が入っている。

`GroupBy.last()` は「グループの最後の行」ではなく、
**列ごとに最後の非null値**を返す。だから

- `status` は最新行の `cancelled`
- `amount` は最新行が欠損なので、**1つ前の行の 2400** を拾う

という、**どの行にも存在しない合成行**ができる。

`B` は行そのものを1本選ぶので、`amount` は欠損のまま。これが正しい。

> この違いは、後続で「amount が欠損の行を除外する」という処理をしたときに
> 効いてくる。`A` だと落ちるべき行が生き残る。**quest-01 の罠がこれ。**

`first()` も同じ性質を持つ。

</details>

### 3-2. head(1) / tail(1) / nth(0)

行を選ぶほうの書き方。こちらは合成行を作らない。

**実行する前に、違いを予想する。**

In [ ]:
# A: groupby().tail(1)
a = dup.sort_values("ingested_at").groupby("order_id").tail(1)
a

In [ ]:
# B: groupby().nth(-1)
b = dup.sort_values("ingested_at").groupby("order_id").nth(-1)
b

<details>
<summary>何が起きたか</summary>

どちらも**行を選ぶ**ので、欠損はそのまま残る。`drop_duplicates` と同じ結果。

| 書き方 | 何を返すか | 合成行ができるか |
| --- | --- | --- |
| `.last()` / `.first()` | **列ごとの非null値** | **できる。危険** |
| `.tail(1)` / `.head(1)` | 行 | できない |
| `.nth(0)` / `.nth(-1)` | 行 | できない |
| `drop_duplicates(keep=...)` | 行 | できない |

`tail(1)` は元の index を保つので `reset_index(drop=True)` を足すことが多い。

**「1行選ぶ」つもりなら `.last()` を使わない。** 覚えることはこれだけ。

</details>

### 3-3. 順序が決まらないとどうなるか

`ingested_at` が同着だったら、どの行が選ばれるか。

**実行する前に、違いを予想する。**

In [ ]:
# A: 同着を作る
tie = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 100},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 999},
])
print(tie.sort_values("ingested_at").drop_duplicates("order_id", keep="last"))

In [ ]:
# B: 全順序をつける
tie = pd.DataFrame([
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 100},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 999},
])
print(tie.sort_values(["ingested_at", "amount"]).drop_duplicates("order_id", keep="last"))

<details>
<summary>何が起きたか</summary>

`A` は「たまたま後ろにあったほう」が選ばれる。いまの pandas の
`sort_values` は既定で安定ソート(`kind="quicksort"` でも実際は安定に振る舞う
ことが多い)なので元の順が保たれるが、**保証されていない**。

入力ファイルの順番が変わったり、並列で読んだりすると結果が変わる。
**流すたびに数字が変わる**という、最も厄介な壊れ方になる。

`B` のように第2キーを足して**全順序**にする。
決定的でないパイプラインは、冪等ではない。

</details>

---
## 4. 重複を調べる

### 4-1. duplicated の keep

どれを「重複」と見なすか。

**実行する前に、違いを予想する。**

In [ ]:
# A: keep='first' (既定)
s = pd.Series(["a", "b", "a", "c", "a"])
print(s.duplicated().tolist())
print("重複とされた数:", s.duplicated().sum())

In [ ]:
# B: keep=False
s = pd.Series(["a", "b", "a", "c", "a"])
print(s.duplicated(keep=False).tolist())
print("重複とされた数:", s.duplicated(keep=False).sum())

<details>
<summary>何が起きたか</summary>

- `keep="first"` は**2回目以降**を True にする(最初は残す)
- `keep="last"` は**最後以外**を True
- `keep=False` は**重複しているものを全部** True

**調査したいときは `keep=False`。** 「重複している値をすべて見たい」のに
既定のまま使うと、最初の1件が見えない。

```python
df[df.duplicated("order_id", keep=False)].sort_values("order_id")
```

これが「重複を目で確認する」ときの定番。

</details>

### 4-2. subset を指定するかどうか

行全体の重複か、キーの重複か。

**実行する前に、違いを予想する。**

In [ ]:
# A: subset なし
a = dup.drop_duplicates()
print("行数:", len(a))
a

In [ ]:
# B: subset='order_id'
b = dup.drop_duplicates(subset="order_id")
print("行数:", len(b))
b

<details>
<summary>何が起きたか</summary>

- `A` は**全列が一致する行**だけを重複と見なす。再送で完全に同じ行が
  来たときに効く。`dup` には完全一致が無いので1行も減らない
- `B` は `order_id` が同じなら重複と見なす。**訂正を畳むならこちら**

どちらが要るかはデータの汚れ方で決まる。
**両方要ることも多い**(完全重複を落としてから、キーで最新を選ぶ)。

</details>

---
## 5. 並べ替え

### 5-1. na_position

欠損はどこに行くか。

**実行する前に、違いを予想する。**

In [ ]:
# A: 既定 (na_position='last')
a = df.sort_values("amount")
a[["order_id", "amount"]]

In [ ]:
# B: na_position='first'
b = df.sort_values("amount", na_position="first")
b[["order_id", "amount"]]

<details>
<summary>何が起きたか</summary>

**昇順でも降順でも、既定では欠損は最後に来る。**
`ascending=False` にしても欠損は末尾のまま。

「最大値の行を取る」つもりで `sort_values(ascending=False).head(1)` と
書くと、全部欠損の列では**欠損の行ではなく最大の行**が取れる。これは
たいてい望ましいが、`na_position="first"` にすると逆になる。挙動を知っておく。

複数列で向きを変えたいときはリストで渡す。

```python
df.sort_values(["region", "amount"], ascending=[True, False])
```

</details>

### 5-2. rank の method

同順位をどう扱うか。SQL の `RANK` / `DENSE_RANK` / `ROW_NUMBER` に対応する。

**実行する前に、違いを予想する。**

In [ ]:
# A: min と dense
s = pd.Series([10, 20, 20, 30])
print("min   :", s.rank(method="min").tolist())
print("dense :", s.rank(method="dense").tolist())

In [ ]:
# B: first と average
s = pd.Series([10, 20, 20, 30])
print("first   :", s.rank(method="first").tolist())
print("average :", s.rank(method="average").tolist())

<details>
<summary>何が起きたか</summary>

| method | 同順位の扱い | SQL |
| --- | --- | --- |
| `min` | 同じ順位。次は飛ぶ (1,2,2,4) | `RANK()` |
| `dense` | 同じ順位。次は飛ばない (1,2,2,3) | `DENSE_RANK()` |
| `first` | 出現順に別々の順位 (1,2,3,4) | `ROW_NUMBER()` |
| `average` | **既定**。平均 (1,2.5,2.5,4) | — |

**既定が `average` なので、何も指定しないと小数が出る。**
「順位」が欲しいときはたいてい `min` か `dense`。

`method="first"` を `groupby` と組み合わせると、SQL の
`ROW_NUMBER() OVER (PARTITION BY ... ORDER BY ...)` と同じことができる。

</details>

---
## 練習

In [ ]:
# 練習1: region ごとに、注文件数・金額合計・金額平均を出す。
#        列名は n_orders / total / avg。region は列として残す。
#        (region が欠損の行は無いが、customer_id が欠損の行も数に入れること)

ans = ...   # ここに書く

assert list(ans.columns) == ["region", "n_orders", "total", "avg"], list(ans.columns)
assert len(ans) == 4
assert ans.set_index("region").loc["east", "n_orders"] == 4
assert ans["total"].sum() == 23220, f"合計が合わない: {ans['total'].sum()}"
print("OK")

In [ ]:
# 練習2: 下の raw から「order_id ごとに ingested_at が最新の1行」を取り出す。
#        合成行を作らないこと。並び順は order_id 昇順、index は振り直す。

raw = pd.DataFrame([
    {"order_id": "O-002", "ingested_at": "2024-02-01 09:00", "amount": 980,  "status": "pending"},
    {"order_id": "O-001", "ingested_at": "2024-02-01 10:00", "amount": 2400, "status": "completed"},
    {"order_id": "O-001", "ingested_at": "2024-02-02 10:00", "amount": None, "status": "cancelled"},
    {"order_id": "O-003", "ingested_at": "2024-02-01 11:00", "amount": 3600, "status": "completed"},
])
raw["amount"] = raw["amount"].astype("Int64")

ans = ...   # ここに書く

assert len(ans) == 3, f"3行のはず: {len(ans)}"
assert ans["order_id"].tolist() == ["O-001", "O-002", "O-003"]
assert pd.isna(ans.loc[0, "amount"]), "O-001 の amount は欠損のまま残るはず (合成行になっている)"
assert ans.loc[0, "status"] == "cancelled"
print("OK")

In [ ]:
# 練習3: 練習2の ans から、amount が欠損している行を除外する。
#        (この順序が大事。先に除外すると O-001 の古い行が生き残ってしまう)

ans3 = ...   # ここに書く

assert ans3["order_id"].tolist() == ["O-002", "O-003"], ans3["order_id"].tolist()
print("OK")

---
## まとめ

| 書き方 | 意味 | 注意 |
| --- | --- | --- |
| `groupby(as_index=False)` | キーを列として残す | 下流に渡すならこちら |
| `groupby(dropna=False)` | キーが欠損の群も残す | **既定は消える。合計が変わる** |
| `size` / `count` | 行数 / 非欠損の数 | `COUNT(*)` と `COUNT(col)` |
| `agg(name=(col, fn))` | 名前付き集約 | 列名を自分で決められる |
| `transform` | 集約値を元の行に配る | 行数が変わらない |
| **`.last()` / `.first()`** | **列ごとの非null値** | **合成行ができる。使わない** |
| `.tail(1)` / `.nth(-1)` | 行を選ぶ | 安全 |
| `drop_duplicates(subset, keep)` | 行を選ぶ | 安全。第一候補 |
| `duplicated(keep=False)` | 重複を全部 True | 調査するとき |
| `sort_values(na_position=)` | 欠損の位置 | 既定は末尾 |
| `rank(method=)` | 同順位の扱い | **既定は average で小数が出る** |

### 「キーごとに最新の1行」の定型

```python
(df.sort_values(["ingested_at", "order_id"])          # 全順序にする
   .drop_duplicates(subset="order_id", keep="last")   # 行を選ぶ
   .reset_index(drop=True))
```

**選んでから、捨てる。** 除外はこの後。

次: `sql-01-select-null-join.ipynb`